# 1. Setup and Load Forecast Data

In [1]:
from pathlib import Path
import gc

import duckdb
import numpy as np
import pandas as pd
import plotly.express as px

In [2]:
current_dir = Path.cwd().resolve()
project_name = 'ride-hailing-demand-fleet-allocation'

if current_dir.name == project_name:
    project_root = current_dir
elif current_dir.name == 'notebooks' and current_dir.parent.name == project_name:
    project_root = current_dir.parent
else:
    project_root = current_dir / project_name

database_path = project_root / 'data' / 'processed' / 'nyc_taxi.db'

if not database_path.exists():
    raise FileNotFoundError(f'Database not found: {database_path}')

temp_dir = project_root / 'data' / 'temp'
temp_dir.mkdir(parents=True, exist_ok=True)

con = duckdb.connect(str(database_path), read_only=True)
con.execute("SET memory_limit = '1GB'")
con.execute("SET threads = 2")
con.execute(f"SET temp_directory = '{temp_dir.as_posix()}'")

In [3]:
df_forecast = con.execute("""
    SELECT *
    FROM forecast_output
    ORDER BY forecast_time, LocationID
""").df()

df_model_evaluation = con.execute("""
    SELECT *
    FROM forecast_model_evaluation
""").df()

df_tier_evaluation = con.execute("""
    SELECT *
    FROM forecast_test_tier_evaluation
    ORDER BY volume_tier
""").df()

display(df_forecast.head())
display(df_model_evaluation.round(2))
display(df_tier_evaluation.round(2))

,forecast_time,LocationID,pickup_zone,pickup_borough,forecast_date,hour_of_day,day_of_week,is_weekend,forecast_completed_trips,volume_tier,model
0,2025-01-01,2,Jamaica Bay,Queens,2025-01-01,0,3,0,0.000000,Low,Historical Baseline
1,2025-01-01,3,Allerton/Pelham Gardens,Bronx,2025-01-01,0,3,0,24.423077,Medium,Historical Baseline
2,2025-01-01,4,Alphabet City,Manhattan,2025-01-01,0,3,0,58.653846,Medium,Historical Baseline
3,2025-01-01,5,Arden Heights,Staten Island,2025-01-01,0,3,0,2.153846,Low,Historical Baseline
4,2025-01-01,6,Arrochar/Fort Wadsworth,Staten Island,2025-01-01,0,3,0,5.673077,Low,Historical Baseline


,evaluation_period,model,WAPE (%),MAE,MAPE nonzero (%),Bias (%)
0,Validation,Historical Baseline,15.40,16.31,24.09,-1.32
1,Validation,HistGradientBoosting,16.89,17.89,27.90,5.15
2,Validation,Linear Regression,18.62,19.71,28.63,-4.22
3,Test,Historical Baseline,20.22,21.85,28.64,-4.81


,volume_tier,WAPE (%),MAE,Bias (%)
0,High,20.53,43.62,-3.26
1,Low,23.65,5.78,-8.40
2,Medium,18.50,16.33,-7.53


In [4]:
print(f'Forecast rows        : {len(df_forecast):,}')
print(f'Model evaluation rows: {len(df_model_evaluation):,}')
print(f'Tier evaluation rows : {len(df_tier_evaluation):,}')
print(f'Forecast period      : {df_forecast["forecast_time"].min()} to {df_forecast["forecast_time"].max()}')
print(f'Forecast model       : {df_forecast["model"].unique().tolist()}')

Forecast rows        : 44,016
Model evaluation rows: 4
Tier evaluation rows : 3
Forecast period      : 2025-01-01 00:00:00 to 2025-01-07 23:00:00
Forecast model       : ['Historical Baseline']


# 2. Validate Forecast Inputs

In [5]:
expected_rows = 7 * 24 * 262
unique_hours = df_forecast['forecast_time'].nunique()
unique_zones = df_forecast['LocationID'].nunique()
duplicate_rows = df_forecast.duplicated(['forecast_time', 'LocationID']).sum()
missing_predictions = df_forecast['forecast_completed_trips'].isna().sum()
negative_predictions = (df_forecast['forecast_completed_trips'] < 0).sum()

zones_per_hour = df_forecast.groupby('forecast_time')['LocationID'].nunique()

print(f'Expected rows       : {expected_rows:,}')
print(f'Actual rows         : {len(df_forecast):,}')
print(f'Unique hours        : {unique_hours:,}')
print(f'Unique zones        : {unique_zones:,}')
print(f'Min zones per hour  : {zones_per_hour.min():,}')
print(f'Max zones per hour  : {zones_per_hour.max():,}')
print(f'Duplicate rows      : {duplicate_rows:,}')
print(f'Missing predictions : {missing_predictions:,}')
print(f'Negative predictions: {negative_predictions:,}')

Expected rows       : 44,016
Actual rows         : 44,016
Unique hours        : 168
Unique zones        : 262
Min zones per hour  : 262
Max zones per hour  : 262
Duplicate rows      : 0
Missing predictions : 0
Negative predictions: 0


In [6]:
df_selected_test = df_model_evaluation[
    df_model_evaluation['evaluation_period'] == 'Test'
]

display(df_selected_test.round(2))
display(df_tier_evaluation.round(2))

,evaluation_period,model,WAPE (%),MAE,MAPE nonzero (%),Bias (%)
3,Test,Historical Baseline,20.22,21.85,28.64,-4.81


,volume_tier,WAPE (%),MAE,Bias (%)
0,High,20.53,43.62,-3.26
1,Low,23.65,5.78,-8.40
2,Medium,18.50,16.33,-7.53


# 3. Build Historical Duration Proxy

In [7]:
# Aggregate historical trip duration
df_duration_base = con.execute("""
    SELECT
        trip.PULocationID AS LocationID,
        EXTRACT(HOUR FROM trip.pickup_datetime) AS hour_of_day,
        ISODOW(trip.pickup_datetime) AS day_of_week,
        COUNT(*) AS trip_count,
        SUM(trip.trip_time) AS total_trip_seconds
    FROM fact_trip AS trip
    JOIN dim_zone AS zone
        ON trip.PULocationID = zone.LocationID
    WHERE zone.LocationID NOT IN (264, 265)
      AND zone.Borough <> 'EWR'
    GROUP BY
        trip.PULocationID,
        hour_of_day,
        day_of_week
""").df()

df_duration_base.head()

,LocationID,hour_of_day,day_of_week,trip_count,total_trip_seconds
0,161,0,1,5626,6473790.0
1,79,0,1,15178,15835823.0
2,148,0,1,13233,14192626.0
3,255,0,1,11900,12145923.0
4,213,0,1,3865,3191452.0


In [8]:
df_duration_lookup = df_duration_base.copy()
df_duration_lookup['historical_avg_trip_minutes'] = (
    df_duration_lookup['total_trip_seconds']
    / df_duration_lookup['trip_count']
    / 60
)

df_duration_lookup = df_duration_lookup[
    ['LocationID', 'hour_of_day', 'day_of_week', 'historical_avg_trip_minutes']
]

In [9]:
df_zone_hour_duration = (
    df_duration_base
    .groupby(['LocationID', 'hour_of_day'], as_index=False)
    .agg(
        trip_count=('trip_count', 'sum'),
        total_trip_seconds=('total_trip_seconds', 'sum')
    )
)

df_zone_hour_duration['zone_hour_avg_minutes'] = (
    df_zone_hour_duration['total_trip_seconds']
    / df_zone_hour_duration['trip_count']
    / 60
)

df_zone_hour_duration = df_zone_hour_duration[
    ['LocationID', 'hour_of_day', 'zone_hour_avg_minutes']
]

In [10]:
df_zone_duration = (
    df_duration_base
    .groupby('LocationID', as_index=False)
    .agg(
        trip_count=('trip_count', 'sum'),
        total_trip_seconds=('total_trip_seconds', 'sum')
    )
)

df_zone_duration['zone_avg_minutes'] = (
    df_zone_duration['total_trip_seconds']
    / df_zone_duration['trip_count']
    / 60
)

df_zone_duration = df_zone_duration[
    ['LocationID', 'zone_avg_minutes']
]

In [11]:
global_avg_trip_minutes = (
    df_duration_base['total_trip_seconds'].sum()
    / df_duration_base['trip_count'].sum()
    / 60
)

In [12]:
print(f'Zone-hour-day lookup rows: {len(df_duration_lookup):,}')
print(f'Zone-hour lookup rows    : {len(df_zone_hour_duration):,}')
print(f'Zone lookup rows         : {len(df_zone_duration):,}')
print(f'Global average duration  : {global_avg_trip_minutes:.2f} minutes')

Zone-hour-day lookup rows: 43,354
Zone-hour lookup rows    : 6,227
Zone lookup rows         : 260
Global average duration  : 20.13 minutes


In [13]:
con.close()
del df_duration_base
gc.collect()

print('Historical source released from memory.')

Historical source released from memory.


# 4. Estimate Forecasted Trip Activity

In [14]:
df_activity = (
    df_forecast
    .merge(
        df_duration_lookup,
        on=['LocationID', 'hour_of_day', 'day_of_week'],
        how='left'
    )
    .merge(
        df_zone_hour_duration,
        on=['LocationID', 'hour_of_day'],
        how='left'
    )
    .merge(
        df_zone_duration,
        on='LocationID',
        how='left'
    )
)

In [15]:
df_activity['duration_source'] = np.select(
    [
        df_activity['historical_avg_trip_minutes'].notna(),
        df_activity['zone_hour_avg_minutes'].notna(),
        df_activity['zone_avg_minutes'].notna()
    ],
    ['zone_hour_day', 'zone_hour', 'zone'],
    default='global'
)

df_activity['historical_avg_trip_minutes'] = (
    df_activity['historical_avg_trip_minutes']
    .fillna(df_activity['zone_hour_avg_minutes'])
    .fillna(df_activity['zone_avg_minutes'])
    .fillna(global_avg_trip_minutes)
)

df_activity = df_activity.drop(
    columns=['zone_hour_avg_minutes', 'zone_avg_minutes']
)

In [16]:
df_activity['forecasted_completed_trip_hours'] = (
    df_activity['forecast_completed_trips']
    * df_activity['historical_avg_trip_minutes']
    / 60
)

df_activity[
    [
        'forecast_time',
        'LocationID',
        'pickup_zone',
        'forecast_completed_trips',
        'historical_avg_trip_minutes',
        'forecasted_completed_trip_hours',
        'duration_source'
    ]
].head().round({
    'forecast_completed_trips': 2,
    'historical_avg_trip_minutes': 2,
    'forecasted_completed_trip_hours': 2
})

,forecast_time,LocationID,pickup_zone,forecast_completed_trips,historical_avg_trip_minutes,forecasted_completed_trip_hours,duration_source
0,2025-01-01,2,Jamaica Bay,0.00,7.40,0.00,zone_hour
1,2025-01-01,3,Allerton/Pelham Gardens,24.42,13.39,5.45,zone_hour_day
2,2025-01-01,4,Alphabet City,58.65,18.26,17.85,zone_hour_day
3,2025-01-01,5,Arden Heights,2.15,14.79,0.53,zone_hour_day
4,2025-01-01,6,Arrochar/Fort Wadsworth,5.67,12.38,1.17,zone_hour_day


In [17]:
duration_source_summary = (
    df_activity['duration_source']
    .value_counts()
    .rename_axis('duration_source')
    .reset_index(name='rows')
)

print(f'Activity rows              : {len(df_activity):,}')
print(f'Missing duration values    : {df_activity["historical_avg_trip_minutes"].isna().sum():,}')
print(f'Missing activity values    : {df_activity["forecasted_completed_trip_hours"].isna().sum():,}')
print(f'Negative activity values   : {(df_activity["forecasted_completed_trip_hours"] < 0).sum():,}')
display(duration_source_summary)

Activity rows              : 44,016
Missing duration values    : 0
Missing activity values    : 0
Negative activity values   : 0


,duration_source,rows
0,zone_hour_day,43354
1,global,336
2,zone_hour,235
3,zone,91


# 5. Calculate Relative Fleet Share

In [18]:
df_positioning = df_activity.copy()

df_positioning['hourly_total_trip_hours'] = (
    df_positioning
    .groupby('forecast_time')['forecasted_completed_trip_hours']
    .transform('sum')
)

df_positioning['recommended_positioning_share_pct'] = (
    df_positioning['forecasted_completed_trip_hours']
    / df_positioning['hourly_total_trip_hours']
    * 100
)

In [19]:
df_hourly_share_check = (
    df_positioning
    .groupby('forecast_time', as_index=False)
    .agg(
        total_positioning_share=('recommended_positioning_share_pct', 'sum'),
        total_trip_hours=('forecasted_completed_trip_hours', 'sum')
    )
)

print(f'Positioning rows       : {len(df_positioning):,}')
print(f'Minimum hourly share  : {df_hourly_share_check["total_positioning_share"].min():.4f}%')
print(f'Maximum hourly share  : {df_hourly_share_check["total_positioning_share"].max():.4f}%')
print(f'Missing shares        : {df_positioning["recommended_positioning_share_pct"].isna().sum():,}')

Positioning rows       : 44,016
Minimum hourly share  : 100.0000%
Maximum hourly share  : 100.0000%
Missing shares        : 0


In [20]:
peak_forecast_time = (
    df_hourly_share_check
    .sort_values('total_trip_hours', ascending=False)
    .iloc[0]['forecast_time']
)

df_peak_positioning = (
    df_positioning[
        df_positioning['forecast_time'] == peak_forecast_time
    ]
    .sort_values('recommended_positioning_share_pct', ascending=False)
    .head(10)
)

print(f'Peak forecast workload: {peak_forecast_time}')

df_peak_positioning[
    [
        'pickup_zone',
        'pickup_borough',
        'forecast_completed_trips',
        'forecasted_completed_trip_hours',
        'recommended_positioning_share_pct'
    ]
].round(2)

Peak forecast workload: 2025-01-04 18:00:00


,pickup_zone,pickup_borough,forecast_completed_trips,forecasted_completed_trip_hours,recommended_positioning_share_pct
23710,JFK Airport,Queens,548.33,452.35,2.89
23716,LaGuardia Airport,Queens,514.62,286.54,1.83
23833,Williamsburg (North Side),Brooklyn,692.54,269.23,1.72
23657,East Village,Manhattan,672.23,228.89,1.46
23639,Crown Heights North,Brooklyn,641.17,214.57,1.37
23646,East Chelsea,Manhattan,496.10,214.56,1.37
23824,West Chelsea/Hudson Yards,Manhattan,526.31,213.62,1.37
23690,Greenpoint,Brooklyn,573.69,198.03,1.27
23809,TriBeCa/Civic Center,Manhattan,545.65,192.61,1.23
23759,Park Slope,Brooklyn,551.06,192.23,1.23


# 6. Create Positioning Priorities

In [21]:
df_recommendation = df_positioning.sort_values(
    ['forecast_time', 'recommended_positioning_share_pct'],
    ascending=[True, False]
).copy()

df_recommendation['positioning_rank'] = (
    df_recommendation
    .groupby('forecast_time')['recommended_positioning_share_pct']
    .rank(method='first', ascending=False)
    .astype('int16')
)

df_recommendation['cumulative_share_pct'] = (
    df_recommendation
    .groupby('forecast_time')['recommended_positioning_share_pct']
    .cumsum()
)

share_before_zone = (
    df_recommendation['cumulative_share_pct']
    - df_recommendation['recommended_positioning_share_pct']
)

df_recommendation['positioning_priority'] = np.select(
    [share_before_zone < 50, share_before_zone < 80],
    ['High', 'Medium'],
    default='Standard'
)

In [22]:
final_cumulative_share = (
    df_recommendation
    .groupby('forecast_time')['cumulative_share_pct']
    .max()
)

df_priority_summary = (
    df_recommendation
    .groupby('positioning_priority', as_index=False)
    .agg(
        total_rows=('LocationID', 'size'),
        avg_positioning_share_pct=('recommended_positioning_share_pct', 'mean')
    )
)

display(df_priority_summary.round(2))

,positioning_priority,total_rows,avg_positioning_share_pct
0,High,7960,1.06
1,Medium,10868,0.46
2,Standard,25188,0.13


In [23]:
df_peak_priority = (
    df_recommendation[
        df_recommendation['forecast_time'].eq(peak_forecast_time)
    ]
    .sort_values('positioning_rank')
    .reset_index(drop=True)
)

print(f'Peak time        : {peak_forecast_time}')
print(f'Zones visualized : {len(df_peak_priority):,}')
print(f'Plot data memory : {df_peak_priority.memory_usage(deep=True).sum() / 1024**2:.2f} MB')

assert len(df_peak_priority) == 262

Peak time        : 2025-01-04 18:00:00
Zones visualized : 262
Plot data memory : 0.12 MB


In [24]:
display(
    df_peak_priority[
        [
            'positioning_rank',
            'pickup_zone',
            'pickup_borough',
            'forecast_completed_trips',
            'forecasted_completed_trip_hours',
            'recommended_positioning_share_pct',
            'cumulative_share_pct',
            'positioning_priority'
        ]
    ]
    .head(15)
    .round(2)
)

,positioning_rank,pickup_zone,pickup_borough,forecast_completed_trips,forecasted_completed_trip_hours,recommended_positioning_share_pct,cumulative_share_pct,positioning_priority
0,1,JFK Airport,Queens,548.33,452.35,2.89,2.89,High
1,2,LaGuardia Airport,Queens,514.62,286.54,1.83,4.72,High
2,3,Williamsburg (North Side),Brooklyn,692.54,269.23,1.72,6.45,High
3,4,East Village,Manhattan,672.23,228.89,1.46,7.91,High
4,5,Crown Heights North,Brooklyn,641.17,214.57,1.37,9.28,High
5,6,East Chelsea,Manhattan,496.10,214.56,1.37,10.65,High
6,7,West Chelsea/Hudson Yards,Manhattan,526.31,213.62,1.37,12.02,High
7,8,Greenpoint,Brooklyn,573.69,198.03,1.27,13.29,High
8,9,TriBeCa/Civic Center,Manhattan,545.65,192.61,1.23,14.52,High
9,10,Park Slope,Brooklyn,551.06,192.23,1.23,15.75,High


In [27]:
fig = px.bar(
    df_peak_priority,
    x='positioning_rank',
    y='recommended_positioning_share_pct',
    color='positioning_priority',
    hover_name='pickup_zone',
    hover_data={
        'pickup_borough': True,
        'forecast_completed_trips': ':.2f',
        'forecasted_completed_trip_hours': ':.2f',
        'recommended_positioning_share_pct': ':.3f',
        'positioning_rank': True
    },
    color_discrete_map={
        'High': '#d95f02',
        'Medium': '#e6ab02',
        'Standard': '#7570b3'
    },
    category_orders={
        'positioning_priority': ['High', 'Medium', 'Standard']
    },
    title=f'Positioning Share of All Zones - {peak_forecast_time}'
)

fig.update_layout(
    width=1000,
    height=500,
    bargap=0,
    margin=dict(l=70, r=40, t=100, b=60),
    xaxis_title='Zone Positioning Rank',
    yaxis_title='Recommended Positioning Share (%)',
    legend=dict(
        title='Positioning Priority',
        orientation='h',
        x=0,
        y=1.12
    )
)

fig.update_traces(marker_line_width=0)
fig.show(renderer='iframe')

# 7. Add Reliability and Limitation Flags

In [28]:
df_tier_metrics = df_tier_evaluation.rename(columns={
    'WAPE (%)': 'test_wape_pct',
    'MAE': 'test_mae',
    'Bias (%)': 'test_bias_pct'
})

df_recommendation = df_recommendation.merge(
    df_tier_metrics[
        ['volume_tier', 'test_wape_pct', 'test_mae', 'test_bias_pct']
    ],
    on='volume_tier',
    how='left'
)

In [29]:
df_recommendation['forecast_reliability'] = np.select(
    [
        df_recommendation['test_wape_pct'] <= 20,
        df_recommendation['test_wape_pct'] <= 22
    ],
    ['Higher', 'Moderate'],
    default='Lower'
)

df_recommendation['duration_reliability'] = np.select(
    [
        df_recommendation['duration_source'] == 'zone_hour_day',
        df_recommendation['duration_source'].isin(['zone_hour', 'zone'])
    ],
    ['Specific historical pattern', 'Broader historical fallback'],
    default='Global duration fallback'
)

In [30]:
is_new_year = (
    df_recommendation['forecast_time']
    .dt.normalize()
    .eq(pd.Timestamp('2025-01-01'))
)

df_recommendation['calendar_warning'] = np.where(
    is_new_year,
    'Holiday pattern not modeled',
    'Regular calendar pattern'
)

In [31]:
needs_review = (
    (df_recommendation['forecast_reliability'] == 'Lower')
    | (df_recommendation['duration_source'] == 'global')
    | is_new_year
)

df_recommendation['review_flag'] = np.where(
    needs_review,
    'Manual review recommended',
    'Standard use'
)

df_recommendation['recommendation_scope'] = 'Relative positioning only'

In [32]:
print(f'Recommendation rows       : {len(df_recommendation):,}')
print(f'Missing tier metrics      : {df_recommendation["test_wape_pct"].isna().sum():,}')
print(f'Missing reliability flags : {df_recommendation["forecast_reliability"].isna().sum():,}')
print(f'Manual review rows        : {(df_recommendation["review_flag"] == "Manual review recommended").sum():,}')

Recommendation rows       : 44,016
Missing tier metrics      : 0
Missing reliability flags : 0
Manual review rows        : 18,960


In [33]:
df_flag_summary = (
    df_recommendation
    .groupby(
        ['forecast_reliability', 'duration_reliability', 'review_flag'],
        as_index=False
    )
    .agg(number_of_rows=('LocationID', 'size'))
)

display(df_flag_summary)

,forecast_reliability,duration_reliability,review_flag,number_of_rows
0,Higher,Specific historical pattern,Manual review recommended,2088
1,Higher,Specific historical pattern,Standard use,12528
2,Lower,Broader historical fallback,Manual review recommended,326
3,Lower,Global duration fallback,Manual review recommended,336
4,Lower,Specific historical pattern,Manual review recommended,14122
5,Moderate,Specific historical pattern,Manual review recommended,2088
6,Moderate,Specific historical pattern,Standard use,12528


In [34]:
df_recommendation = df_recommendation.drop(
    columns='calendar_warning',
    errors='ignore'
)

manual_review = (
    df_recommendation['duration_source'] == 'global'
)

use_with_caution = (
    (df_recommendation['forecast_reliability'] == 'Lower')
    | df_recommendation['duration_source'].isin(['zone_hour', 'zone'])
)

df_recommendation['review_flag'] = np.select(
    [manual_review, use_with_caution],
    ['Manual review recommended', 'Use with caution'],
    default='Standard use'
)

df_recommendation['recommendation_scope'] = 'Relative positioning only'

In [35]:
review_summary = (
    df_recommendation['review_flag']
    .value_counts()
    .rename_axis('review_flag')
    .reset_index(name='number_of_rows')
)

review_summary['row_share_pct'] = (
    review_summary['number_of_rows']
    / len(df_recommendation)
    * 100
)

display(review_summary.round(2))

,review_flag,number_of_rows,row_share_pct
0,Standard use,29232,66.41
1,Use with caution,14448,32.82
2,Manual review recommended,336,0.76


# 8. Save Recommendations

In [37]:
final_columns = [
    'forecast_time', 'forecast_date', 'LocationID',
    'pickup_zone', 'pickup_borough',
    'hour_of_day', 'day_of_week', 'is_weekend',
    'forecast_completed_trips', 'volume_tier', 'model',
    'historical_avg_trip_minutes', 'duration_source',
    'forecasted_completed_trip_hours', 'hourly_total_trip_hours',
    'recommended_positioning_share_pct',
    'positioning_rank', 'cumulative_share_pct',
    'positioning_priority',
    'test_wape_pct', 'test_mae', 'test_bias_pct',
    'forecast_reliability', 'duration_reliability',
    'review_flag', 'recommendation_scope'
]

df_final_recommendation = (
    df_recommendation[final_columns]
    .sort_values(['forecast_time', 'positioning_rank'])
    .reset_index(drop=True)
)

print(f'Final rows   : {len(df_final_recommendation):,}')
print(f'Final columns: {len(df_final_recommendation.columns)}')
print(f'Missing values: {df_final_recommendation.isna().sum().sum():,}')

Final rows   : 44,016
Final columns: 26
Missing values: 0


In [38]:
display(
    df_final_recommendation[
        [
            'forecast_time', 'positioning_rank', 'pickup_zone',
            'forecast_completed_trips',
            'forecasted_completed_trip_hours',
            'recommended_positioning_share_pct',
            'positioning_priority', 'review_flag'
        ]
    ]
    .head(10)
    .round(2)
)

C:\Users\acer\AppData\Local\Temp\ipykernel_11528\2432541784.py:12: UserWarning: obj.round has no effect with datetime, timedelta, or period dtypes. Use obj.dt.round(...) instead.
  .round(2)


,forecast_time,positioning_rank,pickup_zone,forecast_completed_trips,forecasted_completed_trip_hours,recommended_positioning_share_pct,positioning_priority,review_flag
0,2025-01-01,1,JFK Airport,583.48,328.15,7.16,High,Standard use
1,2025-01-01,2,LaGuardia Airport,393.37,153.88,3.36,High,Standard use
2,2025-01-01,3,Times Sq/Theatre District,377.88,123.56,2.69,High,Standard use
3,2025-01-01,4,Midtown Center,349.17,101.90,2.22,High,Standard use
4,2025-01-01,5,East Village,306.58,90.30,1.97,High,Standard use
5,2025-01-01,6,Lower East Side,264.48,77.49,1.69,High,Standard use
6,2025-01-01,7,Midtown South,223.85,74.83,1.63,High,Standard use
7,2025-01-01,8,Clinton East,215.27,73.22,1.60,High,Standard use
8,2025-01-01,9,East Chelsea,206.00,72.71,1.59,High,Standard use
9,2025-01-01,10,Midtown North,200.52,68.89,1.50,High,Standard use


In [39]:
save_con = duckdb.connect(str(database_path))
save_con.register('recommendation_data', df_final_recommendation)

save_con.execute("""
    CREATE OR REPLACE TABLE fleet_allocation_recommendation AS
    SELECT *
    FROM recommendation_data
""")

print('DuckDB table saved: fleet_allocation_recommendation')

DuckDB table saved: fleet_allocation_recommendation


In [40]:
output_dir = project_root / 'outputs'
output_dir.mkdir(parents=True, exist_ok=True)

parquet_path = output_dir / 'fleet_allocation_recommendation.parquet'

save_con.execute(f"""
    COPY fleet_allocation_recommendation
    TO '{parquet_path.as_posix()}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

print(f'Parquet saved: {parquet_path}')

Parquet saved: D:\My Journey\Data Project\ride-hailing-demand-fleet-allocation\outputs\fleet_allocation_recommendation.parquet


In [41]:
saved_rows = save_con.execute("""
    SELECT COUNT(*)
    FROM fleet_allocation_recommendation
""").fetchone()[0]

print(f'Expected rows: {len(df_final_recommendation):,}')
print(f'Saved rows   : {saved_rows:,}')
print(f'Rows match   : {saved_rows == len(df_final_recommendation)}')
print(f'File exists  : {parquet_path.exists()}')

Expected rows: 44,016
Saved rows   : 44,016
Rows match   : True
File exists  : True


# 9. Final Validation and Close Connection

In [43]:
df_database_check = save_con.execute("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT forecast_time) AS unique_hours,
        COUNT(DISTINCT LocationID) AS unique_zones,
        MIN(forecast_time) AS min_forecast_time,
        MAX(forecast_time) AS max_forecast_time,
        SUM(
            CASE WHEN forecast_time IS NULL
                   OR LocationID IS NULL
                   OR forecast_completed_trips IS NULL
                   OR recommended_positioning_share_pct IS NULL
                 THEN 1 ELSE 0 END
        ) AS missing_critical_values,
        SUM(
            CASE WHEN forecast_completed_trips < 0
                 THEN 1 ELSE 0 END
        ) AS negative_forecasts
    FROM fleet_allocation_recommendation
""").df()

display(df_database_check)

,total_rows,unique_hours,unique_zones,min_forecast_time,max_forecast_time,missing_critical_values,negative_forecasts
0,44016,168,262,2025-01-01,2025-01-07 23:00:00,0.0,0.0


In [44]:
duplicate_rows = save_con.execute("""
    SELECT COALESCE(SUM(row_count - 1), 0)
    FROM (
        SELECT COUNT(*) AS row_count
        FROM fleet_allocation_recommendation
        GROUP BY forecast_time, LocationID
        HAVING COUNT(*) > 1
    )
""").fetchone()[0]

print(f'Duplicate rows: {duplicate_rows:,}')

Duplicate rows: 0


In [45]:
df_hourly_final_check = save_con.execute("""
    WITH hourly_check AS (
        SELECT
            forecast_time,
            COUNT(*) AS zone_rows,
            SUM(recommended_positioning_share_pct) AS total_share,
            MIN(positioning_rank) AS min_rank,
            MAX(positioning_rank) AS max_rank
        FROM fleet_allocation_recommendation
        GROUP BY forecast_time
    )
    SELECT
        MIN(zone_rows) AS min_zones,
        MAX(zone_rows) AS max_zones,
        MIN(total_share) AS min_share,
        MAX(total_share) AS max_share,
        SUM(
            CASE WHEN zone_rows <> 262
                   OR ABS(total_share - 100) > 0.0001
                   OR min_rank <> 1
                   OR max_rank <> 262
                 THEN 1 ELSE 0 END
        ) AS invalid_hours
    FROM hourly_check
""").df()

display(df_hourly_final_check.round(4))

,min_zones,max_zones,min_share,max_share,invalid_hours
0,262,262,100.0,100.0,0.0


In [46]:
df_saved_summary = save_con.execute("""
    SELECT
        positioning_priority,
        review_flag,
        COUNT(*) AS number_of_rows
    FROM fleet_allocation_recommendation
    GROUP BY positioning_priority, review_flag
    ORDER BY positioning_priority, review_flag
""").df()

display(df_saved_summary)

,positioning_priority,review_flag,number_of_rows
0,High,Standard use,7954
1,High,Use with caution,6
2,Medium,Standard use,10841
3,Medium,Use with caution,27
4,Standard,Manual review recommended,336
5,Standard,Standard use,10437
6,Standard,Use with caution,14415


In [47]:
save_con.unregister('recommendation_data')
save_con.close()
gc.collect()

print('Recommendation table and Parquet file saved successfully.')
print('Database connection closed.')

Recommendation table and Parquet file saved successfully.
Database connection closed.
